In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [4]:
# ================================
# 🚀 FAST OPTUNA + LGBM PIPELINE
# ================================

import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

print("🚀 Starting OPTUNA pipeline...")

# ================================
# LOAD DATA
# ================================
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv')

target = 'Irrigation_Need'

X = train.drop(columns=[target, 'id'])
y = train[target]

X_test = test.drop(columns=['id'])
test_ids = test['id']

# ================================
# TARGET ENCODING
# ================================
le = LabelEncoder()
y = le.fit_transform(y)

# ================================
# FEATURE ENGINEERING
# ================================
def create_features(df):
    df['ET0'] = df['Temperature_C'] * df['Wind_Speed_kmh'] * df['Sunlight_Hours'] / (df['Humidity'] + 1)
    df['water_balance'] = df['Rainfall_mm'] + df['Previous_Irrigation_mm'] - df['ET0']
    df['moisture_stress'] = df['Temperature_C'] * (100 - df['Humidity']) / (df['Soil_Moisture'] + 1)
    df['soil_retention'] = df['Soil_Moisture'] * df['Organic_Carbon']
    df['crop_stage'] = df['Crop_Type'] + "_" + df['Crop_Growth_Stage']
    return df

X = create_features(X)
X_test = create_features(X_test)

# ================================
# ENCODE CATEGORICALS
# ================================
cat_cols = X.select_dtypes(include='object').columns

for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# ================================
# CV SETUP
# ================================
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ================================
# OPTUNA OBJECTIVE
# ================================
def objective(trial):

    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",
        "boosting_type": "gbdt",
        "verbosity": -1,

        # 🔥 tuned params
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.1),
        "num_leaves": trial.suggest_int("num_leaves", 31, 128),
        "max_depth": trial.suggest_int("max_depth", 4, 10),

        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),

        "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),

        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),

        "n_estimators": 500,
        "n_jobs": -1,
    }

    oof_preds = np.zeros((len(X), 3))

    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model = lgb.LGBMClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric="multi_logloss",
            callbacks=[lgb.early_stopping(50, verbose=False)]
        )

        oof_preds[val_idx] = model.predict_proba(X_val)

    preds = np.argmax(oof_preds, axis=1)
    score = accuracy_score(y, preds)

    return score


# ================================
# RUN OPTUNA (FAST MODE)
# ================================
print("🔍 Running Optuna...")

study = optuna.create_study(direction="maximize")

study.optimize(objective, n_trials=7)  # ⚡ keep small for speed

print("✅ Best score:", study.best_value)
print("🔥 Best params:", study.best_params)

# ================================
# FINAL MODEL)
# ================================
print("🚀 Training final model...")

best_params = study.best_params
best_params.update({
    "objective": "multiclass",
    "num_class": 3,
    "n_estimators": 1000,
    "n_jobs": -1
})

final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y)

# ================================
# PREDICTIONS
# ================================
final_preds = final_model.predict(X_test)
final_preds = le.inverse_transform(final_preds)

submission = pd.DataFrame({
    "id": test_ids,
    "Irrigation_Need": final_preds
})

submission.to_csv("submission.csv", index=False)

print("🎉 DONE! Optuna submission ready")

🚀 Starting OPTUNA pipeline...


[I 2026-04-21 02:07:09,109] A new study created in memory with name: no-name-3751aaa0-b536-41b7-8408-b63e414ef251


🔍 Running Optuna...


[I 2026-04-21 02:10:09,261] Trial 0 finished with value: 0.9843 and parameters: {'learning_rate': 0.05307478413419152, 'num_leaves': 104, 'max_depth': 8, 'feature_fraction': 0.8737885073267216, 'bagging_fraction': 0.8641874366873128, 'bagging_freq': 4, 'min_child_samples': 83, 'lambda_l1': 0.9642797078627391, 'lambda_l2': 1.9010206595714896}. Best is trial 0 with value: 0.9843.
[I 2026-04-21 02:12:18,285] Trial 1 finished with value: 0.9843666666666666 and parameters: {'learning_rate': 0.0855932500127864, 'num_leaves': 93, 'max_depth': 4, 'feature_fraction': 0.8513301228291579, 'bagging_fraction': 0.6750344164931303, 'bagging_freq': 3, 'min_child_samples': 72, 'lambda_l1': 1.887918380993359, 'lambda_l2': 4.376987694450642}. Best is trial 1 with value: 0.9843666666666666.
[I 2026-04-21 02:14:27,317] Trial 2 finished with value: 0.9843 and parameters: {'learning_rate': 0.049115134743048364, 'num_leaves': 111, 'max_depth': 4, 'feature_fraction': 0.7138481187476938, 'bagging_fraction': 0.6

✅ Best score: 0.9843984126984127
🔥 Best params: {'learning_rate': 0.06492195172716828, 'num_leaves': 112, 'max_depth': 6, 'feature_fraction': 0.6368732723325782, 'bagging_fraction': 0.9051567158271968, 'bagging_freq': 6, 'min_child_samples': 58, 'lambda_l1': 4.143901087708352, 'lambda_l2': 1.3839217192590674}
🚀 Training final model...
🎉 DONE! Optuna submission ready
